### RAG with PDF 📄 Data extraction to give context to LLM 🧠

<img src="./Images/RAGs.png" width="800" height="700" style="display: block; margin: auto;">

In [1]:
from dotenv import load_dotenv
load = load_dotenv('./../.env')


In [2]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

### 1. Extracting the PDF files

In [3]:
from langchain_community.document_loaders import PyPDFLoader

pdf1 = "./attention.pdf"
pdf2 = "./LLMForgetting.pdf"
pdf3 = "./TestingAndEvaluatingLLM.pdf"

pdfFiles = [pdf1, pdf2, pdf3]

documents = []

for pdf in pdfFiles:
    loader = PyPDFLoader(pdf)
    documents.extend(loader.load())

print(len(documents))

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_53739/2168121689.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/Type': '/Font', '/Subtype': '/Type1', '/BaseFont': '/TAUCXQ+TimesNewRomanPSMT', '/Encoding': IndirectObject(419, 0, 4667294688), '/FirstChar': 3, '/FontDescriptor': IndirectObject(420, 0, 4667294688), '/LastChar': 31, '/Widths': [500, 333, 500, 278, 944, 333, 556, 500, 667, 500, 722, 389, 444, 722, 278, 611, 500, 722, 722, 500, 278, 444, 889, 500, 500, 500, 500, 250, 500]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF T

253


In [4]:
print(documents[:1])

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': './attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukasz

### 2. Text Splitting into Chunks

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                               chunk_overlap=200, add_start_index=True)

all_splits = text_splitter.split_documents(documents)

len(all_splits)

640

### 3. Embedding

In [6]:
from langchain_community.embeddings import OllamaEmbeddings # Changed based on latest version recommendation

embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")

vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
len(vector_1), len(vector_1)

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_53739/1293922472.py:3: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  embeddings = OllamaEmbeddings(model="nomic-embed-text:v1.5")


(768, 768)

### 4. Vector Stores

In [7]:
from langchain_chroma import Chroma

vector_store = Chroma.from_documents(
    documents = all_splits,
    embedding=embeddings,
    persist_directory="./chroma_langchain_db",  # Where to save data locally, remove if not necessary
)

### 5. Retriving from the Persistant Vector Datastore

In [8]:
from langchain_chroma import Chroma


vector_store = Chroma(persist_directory='./chroma_langchain_db', embedding_function=embeddings)

result = vector_store.similarity_search("What is Bias testing", k=3)

for doc in result:
    print(doc.page_content)

performs bias measurement,i.e.to what degree is the system biased.
Recall that there are two types of bias,i.e.absolute bias and relative
bias. Absolute bias is the bias that a conversational system directly
expresses, while relative bias refers to the system treating different
groups differently. In the following, we first introduce how BiasAsker
measure and quantify two types of bias, respectively.
Absolute Bias Measurement.
We consider that a system exhibits absolute bias if: it expresses
affirmation in response to a yes-no question, or it makes a choice in
response to a choice question, or it provides an explanation to a why-
question. To quantify the degree to which the system is biased and
gain further insight into the biased associations in terms of absolute
bias, we define the following quantities:
Absolute bias rate. We define absolute bias rate as the
percentage of questions that trigger absolute bias among all queries
144 CHAPTER 7. SOCIAL BIAS
The third threat lies in the c

In [9]:
result = vector_store.similarity_search_with_score("What are the types of LLM Testing")

result[0]

(Document(id='af7df205-5fdd-4819-b404-84eca77725a3', metadata={'total_pages': 223, 'page_label': '24', 'trapped': '/False', 'subject': '', 'author': '', 'moddate': '2024-09-04T00:37:21+00:00', 'start_index': 1604, 'page': 42, 'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'title': '', 'creationdate': '2024-09-04T00:37:21+00:00', 'source': './TestingAndEvaluatingLLM.pdf', 'keywords': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5'}, page_content='need testing for reliability and safety. Besides, I highlight that the\ntesting of LLMs has new challenges compared with the testing of\nthe traditional software mentioned above. On the one hand, LLMs\ncan receive any sentence as input, so how to comprehensively and\nsystematically generate input test cases is not trivial. On the other\nhand, LLMs can generate diverse human-like output, which leads to\ndifficulties in detecting failures automatically and accurately

### 6. Retrivers in Langchain

In [10]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs = {"k": 1}
)

retriever.batch(
    [
        "What is the Bias Measurement",
        "How to test human safety against LLM",
        "How LLM forgets the context"
    ]
)


[[Document(id='d7282a9f-e4d3-47ed-b1b5-9ee9db92b509', metadata={'producer': 'pdfTeX-1.40.25', 'subject': '', 'creationdate': '2024-09-04T00:37:21+00:00', 'title': '', 'page': 151, 'page_label': '133', 'keywords': '', 'source': './TestingAndEvaluatingLLM.pdf', 'start_index': 832, 'total_pages': 223, 'moddate': '2024-09-04T00:37:21+00:00', 'creator': 'LaTeX with hyperref', 'author': '', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5'}, page_content='performs bias measurement,i.e.to what degree is the system biased.\nRecall that there are two types of bias,i.e.absolute bias and relative\nbias. Absolute bias is the bias that a conversational system directly\nexpresses, while relative bias refers to the system treating different\ngroups differently. In the following, we first introduce how BiasAsker\nmeasure and quantify two types of bias, respectively.\nAbsolute Bias Measurement.\nWe consider that a system exh

### Document Retrival Manually

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

query = "What exactly does Testing the Factual Correctness of LLM tells"

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

prompt_template = ChatPromptTemplate.from_template(
    """
    You are an AI Assisant. Use the following context to answer the question correctly.
    If you dont know the answer, just tell, I dont know.
    
    Also, summarize the response in MD format
    
    "context: {context} \n\n"
    "question: {question} \n\n"
    "AI answer:
    
    """
)

chain = prompt_template | llm | StrOutputParser()

response = chain.invoke({"context": context_text, "question": query})

print(response)

**Testing the Factual Correctness of LLMs reveals that these models contain factual errors in 45% of their responses. Furthermore, it highlights the potential for improving their accuracy through targeted methods. For example, using the FactChecker approach can enhance the accuracy of models like llama-2-13b-chat from 35.3% to 68.5%. This research is open to the public, encouraging further investigation into LLM evaluation and development to better benefit society. However, the methods used have limitations, including the possibility of false positives and negatives, and cannot fully resolve the accuracy issues.**


### Using Langchain Hub for Prompt
#### Langchain Hub has breaking change in v1.0.0

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchainhub import Client
import json
from langchain_core.load import load

query = "How to test Translation in LLM?"

retrieved_docs = retriever.invoke(query)

context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])


hub = Client()
prompt_data = hub.pull("rlm/rag-prompt")

prompt_dict = json.loads(prompt_data)
prompt = load(prompt_dict)

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"context": context_text, "question": query})

print(response)

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_53739/3765046818.py:14: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  prompt_data = hub.pull("rlm/rag-prompt")
/Users/karthikkk/tryout/LangChainTrainings/.venv/lib/python3.12/site-packages/langchainhub/client.py:326: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = self.pull_repo(owner_repo_commit)
/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_53739/3765046818.py:17: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  prompt = load(prompt_dict)
/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_53739/3765046818.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass 

To test translation in LLMs, translate the query prompt to English using Google Translate and feed it along with the translated response to ChatGPT for evaluation. This method was validated through human annotation on 1,400 cases and comparison with GPT-4 evaluations, achieving an accuracy of 88.5%.


### Retrieving data using RetrievalQA is obsolete now. Use Chains instead.
### Breaking changes in Langchain v1.0.0

In [13]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create prompt
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# Format documents function
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build RAG chain with LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Invoke
response = rag_chain.invoke("What exactly does Testing the Factual Correctness of LLM tells")
print(response)

Testing the factual correctness of LLMs reveals that they contain factual errors in 45% of questions, as identified by FactChecker. This process also highlights opportunities to improve LLM accuracy, such as enhancing the factual correctness of models like llama-2-13b-chat. However, the method has limitations, including potential false positives and negatives, and cannot fully resolve the issues.
